In [117]:
import regionmask, geopandas as gpd, matplotlib.pyplot as plt, cartopy, os, pandas as pd, glob
from IPython.display import clear_output

from shapely.geometry import Point

dataproj = cartopy.crs.PlateCarree()
mapproj = cartopy.crs.Robinson()

# GHCN-D stations

## Download all station metadata

In [2]:
import sys; sys.path.append('/rds/general/user/cb2714/home/get-station-data'); 
from get_station_data import ghcnd
from get_station_data.util import nearest_stn

In [ ]:
# import all metadata
stn_md = ghcnd.get_stn_metadata()

In [28]:
# add geometry
px = gpd.GeoDataFrame(stn_md, geometry = gpd.points_from_xy(stn_md.lon, stn_md.lat), crs = "EPSG:4326")#.clip((-20,-36,54,39))

# filter to only US, Mexico & Canada
px = px.loc[[s[:2] in ["MX", "CA", "US"] for s in px.station]]

In [ ]:
# load station locations

In [186]:
stadia = pd.read_csv("world-cup-stadia.csv")
stadia = gpd.GeoDataFrame(stadia, geometry = gpd.points_from_xy(stadia.lon, stadia.lat), crs = "EPSG:4326")

In [187]:
d = 0.1
px_list = []
for k,r in stadia.iterrows():
    px2 = px.clip((r.lon-d, r.lat-d, r.lon+d, r.lat+d)).copy()
    px2["s_city"] = r.city
    px2["s_lon"] = r.lon
    px2["s_lat"] = r.lat
    px2["s_dist"] = px2.distance(Point(r.lon, r.lat))
    px_list.append(px2)
    clear_output(wait = False) # otherwise MANY error messages when we calculate the distance
px_list = pd.concat(px_list)

In [188]:
px_list = px_list.loc[(px_list.end_year >= 2020) & (px_stad.start_year <= 2000)]

In [ ]:
# loop over stations and download
for i in range(len(px_list)):
    print(i, "/", len(px_list))
    r = px_list.iloc[[i]]
    new_fnm = "ghcnd-stations/stn_"+r.station.values[0]+".csv"
    if os.path.exists(new_fnm): continue

    # faster to download without flags, but no QA - should check this in final station selection
    ghcnd.get_data(r, include_flags = True).to_csv(new_fnm, index = None)
clear_output(wait = False)
print("Done.")

## Check variables

In [ ]:
md_list = []
for fnm in sorted(glob.glob("ghcnd-stations/*.csv")):
    print(fnm)
    df = pd.read_csv(fnm)

    # loop over all available elements
    el = list(set(df.element))
    for e in el:
        edf = df.loc[df.element == e].dropna()
        summ = {"station" : edf.station.values[0], "element" : e, "stn_name" : edf.name.values[0],
                "start_date" : edf.date.min(), "end_date" : edf.date.max(), "ndays" : len(edf)}
        md_list.append(summ)

clear_output(wait = False)

ghcnd-stations/stn_CA001105658.csv
ghcnd-stations/stn_CA001105669.csv
ghcnd-stations/stn_CA001108395.csv
ghcnd-stations/stn_CA001108446.csv
ghcnd-stations/stn_CA001108824.csv
ghcnd-stations/stn_MXM00076393.csv
ghcnd-stations/stn_MXM00076612.csv
ghcnd-stations/stn_MXM00076680.csv


In [223]:
md_list = pd.DataFrame(md_list)

# merge with original station data
px_md = pd.merge(px_list[["s_city", "s_lon", "s_lat", "station", "lat", "lon", "elev", "s_dist"]], md_list, left_on = "station", right_on = "station", how = "outer")
px_md["nyears"] = px_md["ndays"] / 365.25

# filter any variables with less than 10 years of complete data
px_md = px_md.loc[px_md.nyears >= 10]

In [224]:
px_md.to_csv("station-metadata-for-stadia.csv", index = None)